# Module 4, Notebook 2: Bayesian Dynamic Factor Models

## Learning Objectives

By completing this notebook, you will:

1. **Specify** prior distributions for DFM parameters
2. **Implement** a Gibbs sampler for posterior inference
3. **Draw** factors using the simulation smoother
4. **Derive** conditional posterior distributions
5. **Assess** MCMC convergence via diagnostics
6. **Compute** posterior credible intervals and predictive distributions
7. **Compare** Bayesian and frequentist estimates

## Prerequisites

- Module 2: Kalman filter and smoother
- Module 4, Notebook 1: EM algorithm
- Basic Bayesian inference concepts
- Markov Chain Monte Carlo fundamentals

## Estimated Time: 110 minutes

---

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import invgamma, invwishart, multivariate_normal
from scipy.linalg import solve, cholesky
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
%matplotlib inline

print("Setup complete!")
print(f"NumPy version: {np.__version__}")
print(f"SciPy version: {stats.__version__}")

## 1. Bayesian Framework

### Why Bayesian Estimation?

**Frequentist (MLE):**
- Point estimates: $\hat{\theta}_{MLE}$
- Uncertainty via asymptotic theory
- No way to incorporate prior knowledge

**Bayesian:**
- Full distributions: $p(\theta | X)$
- Natural uncertainty quantification
- Prior information + data = posterior
- Regularization through priors

### Bayes' Theorem

$$p(\theta | X) = \frac{p(X | \theta) p(\theta)}{p(X)} \propto p(X | \theta) p(\theta)$$

- **Prior:** $p(\theta)$ - what we believe before seeing data
- **Likelihood:** $p(X | \theta)$ - probability of data given parameters  
- **Posterior:** $p(\theta | X)$ - updated beliefs after seeing data

### The Model

**Measurement:**
$$X_t = \Lambda F_t + e_t, \quad e_t \sim N(0, \Sigma_e)$$

**Dynamics:**
$$F_t = \Phi F_{t-1} + \eta_t, \quad \eta_t \sim N(0, \Sigma_\eta)$$

**Parameters to estimate:**
- $\Lambda$ (N × r loadings)
- $\Phi$ (r × r dynamics)
- $\Sigma_e$ (N × N measurement errors, diagonal)
- $\Sigma_\eta$ (r × r factor innovations, often fixed at $I_r$)
- $F_{1:T}$ (T × r latent factors)

## 2. Prior Specifications

We specify conjugate priors for computational convenience.

In [ ]:
class BayesianPriors:
    """
    Prior specifications for Bayesian DFM.
    """
    
    def __init__(self, n_factors, n_variables, informative=False):
        """
        Initialize prior hyperparameters.
        
        Parameters
        ----------
        n_factors : int
            Number of factors
        n_variables : int
            Number of variables
        informative : bool
            Use informative vs diffuse priors
        """
        self.r = n_factors
        self.N = n_variables
        
        if informative:
            self._set_informative_priors()
        else:
            self._set_diffuse_priors()
    
    def _set_diffuse_priors(self):
        """
        Weakly informative (diffuse) priors.
        
        Use when we have little prior knowledge.
        """
        # Loadings: Normal with large variance
        self.mu_lambda = np.zeros(self.r)
        self.Sigma_lambda = 10 * np.eye(self.r)  # Large variance = diffuse
        
        # Dynamics: Normal, slightly persistent
        self.mu_phi = np.zeros(self.r)
        self.mu_phi[0] = 0.5  # Weak prior for persistence
        self.Sigma_phi = np.eye(self.r)  # Moderate variance
        
        # Idiosyncratic variances: Inverse-Gamma
        # IG(a, b) has mean b/(a-1) for a > 1
        self.a_e = 2.5  # Shape
        self.b_e = 1.5  # Scale -> mean = 1.5/1.5 = 1
        
        # Factor innovations: Fixed for identification
        self.Sigma_eta_fixed = True
        self.Sigma_eta = np.eye(self.r)
    
    def _set_informative_priors(self):
        """
        Informative priors based on economic theory.
        
        Use when we have strong prior beliefs.
        """
        # Loadings: Tighter around zero (sparse-ish)
        self.mu_lambda = np.zeros(self.r)
        self.Sigma_lambda = np.eye(self.r)  # Moderate shrinkage
        
        # Dynamics: Persistence around 0.7
        self.mu_phi = np.zeros(self.r)
        self.mu_phi[0] = 0.7  # Strong prior for persistence
        self.Sigma_phi = 0.1 * np.eye(self.r)  # Tight variance
        
        # Idiosyncratic variances: Tighter around 0.5
        self.a_e = 5.0
        self.b_e = 2.0  # Mean = 0.5
        
        # Factor innovations: Fixed
        self.Sigma_eta_fixed = True
        self.Sigma_eta = np.eye(self.r)
    
    def summary(self):
        """Print prior summary."""
        print("="*60)
        print("PRIOR SPECIFICATIONS")
        print("="*60)
        print(f"\nLoadings (Lambda):")
        print(f"  Distribution: N(mu_lambda, Sigma_lambda)")
        print(f"  Prior mean: {self.mu_lambda}")
        print(f"  Prior variance (diagonal): {np.diag(self.Sigma_lambda)}")
        
        print(f"\nDynamics (Phi):")
        print(f"  Distribution: N(mu_phi, Sigma_phi)")
        print(f"  Prior mean: {self.mu_phi}")
        print(f"  Prior variance (diagonal): {np.diag(self.Sigma_phi)}")
        
        print(f"\nIdiosyncratic Errors (sigma_e^2):")
        print(f"  Distribution: Inverse-Gamma(a_e, b_e)")
        print(f"  Shape (a_e): {self.a_e}")
        print(f"  Scale (b_e): {self.b_e}")
        prior_mean = self.b_e / (self.a_e - 1) if self.a_e > 1 else np.inf
        print(f"  Prior mean: {prior_mean:.3f}")
        
        print(f"\nFactor Innovations (Sigma_eta):")
        if self.Sigma_eta_fixed:
            print(f"  Fixed at I_r for identification")
        print("="*60)


# Demonstrate priors
print("Diffuse Priors:")
priors_diffuse = BayesianPriors(n_factors=2, n_variables=10, informative=False)
priors_diffuse.summary()

print("\n\nInformative Priors:")
priors_inform = BayesianPriors(n_factors=2, n_variables=10, informative=True)
priors_inform.summary()

### Visualize Prior Distributions

In [ ]:
# Purpose: Visualize what the priors look like

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Prior on loading (normal)
ax = axes[0, 0]
x = np.linspace(-5, 5, 1000)
y_diffuse = stats.norm.pdf(x, 0, np.sqrt(10))  # Diffuse
y_inform = stats.norm.pdf(x, 0, 1)  # Informative

ax.plot(x, y_diffuse, linewidth=2.5, label='Diffuse (var=10)')
ax.plot(x, y_inform, linewidth=2.5, label='Informative (var=1)')
ax.set_xlabel('Loading Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Prior on Loadings', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Prior on dynamics parameter (normal)
ax = axes[0, 1]
x = np.linspace(-0.5, 1.5, 1000)
y_diffuse = stats.norm.pdf(x, 0.5, 1.0)
y_inform = stats.norm.pdf(x, 0.7, 0.32)  # sqrt(0.1)

ax.plot(x, y_diffuse, linewidth=2.5, label='Diffuse (mean=0.5, var=1)')
ax.plot(x, y_inform, linewidth=2.5, label='Informative (mean=0.7, var=0.1)')
ax.set_xlabel('Phi Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Prior on Dynamics Phi[0,0]', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Prior on variance (inverse-gamma)
ax = axes[1, 0]
x = np.linspace(0.01, 5, 1000)
y_diffuse = invgamma.pdf(x, a=2.5, scale=1.5)
y_inform = invgamma.pdf(x, a=5.0, scale=2.0)

ax.plot(x, y_diffuse, linewidth=2.5, label='Diffuse (a=2.5, b=1.5)')
ax.plot(x, y_inform, linewidth=2.5, label='Informative (a=5, b=2)')
ax.axvline(1.0, color='gray', linestyle=':', linewidth=2, alpha=0.5)
ax.set_xlabel('Variance', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Prior on Idiosyncratic Variance', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Comparison of concentration
ax = axes[1, 1]
categories = ['Loadings', 'Dynamics', 'Variances']
diffuse_strength = [1/10, 1/1.0, 1.5/2.5]  # Inverse variance = precision
inform_strength = [1/1, 1/0.1, 2.0/5.0]

x_pos = np.arange(len(categories))
width = 0.35

ax.bar(x_pos - width/2, diffuse_strength, width, label='Diffuse', alpha=0.8)
ax.bar(x_pos + width/2, inform_strength, width, label='Informative', alpha=0.8)
ax.set_xlabel('Parameter Type', fontsize=12)
ax.set_ylabel('Prior Precision (1/variance)', fontsize=12)
ax.set_title('Prior Strength Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(categories)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey insight: Informative priors are more concentrated (higher precision).")
print("This regularizes estimates, preventing overfitting.")

## 3. The Gibbs Sampler

The Gibbs sampler iteratively samples from conditional posteriors:

1. **Draw factors:** $F_{1:T} | X, \Lambda, \Phi, \Sigma_e, \Sigma_\eta$ (simulation smoother)
2. **Draw loadings:** $\Lambda | X, F, \Sigma_e$ (multivariate normal)
3. **Draw dynamics:** $\Phi | F, \Sigma_\eta$ (multivariate normal)
4. **Draw variances:** $\Sigma_e | X, F, \Lambda$ (inverse-gamma)

Let's implement each conditional distribution.

### Conditional 1: Factors Given Everything

This requires the **simulation smoother**.

In [ ]:
def simulation_smoother_simple(X, Lambda, Phi, Sigma_e, Sigma_eta):
    """
    Draw factors from p(F | X, theta) using simulation smoother.
    
    Simplified version for AR(1) factors.
    
    Parameters
    ----------
    X : ndarray (T, N)
        Observations
    Lambda : ndarray (N, r)
        Loadings
    Phi : ndarray (r, r)
        Factor dynamics
    Sigma_e : ndarray (N, N)
        Measurement error covariance
    Sigma_eta : ndarray (r, r)
        Factor innovation covariance
        
    Returns
    -------
    F : ndarray (T, r)
        Simulated factor path
    """
    T, N = X.shape
    r = Lambda.shape[1]
    
    # Step 1: Run Kalman filter
    F_filt = np.zeros((T, r))
    P_filt = np.zeros((T, r, r))
    F_pred = np.zeros((T, r))
    P_pred = np.zeros((T, r, r))
    
    # Initialize
    F_pred[0] = np.zeros(r)
    
    # Stationary variance: P = Phi P Phi' + Q
    try:
        Phi_kron = np.kron(Phi, Phi)
        vec_Q = Sigma_eta.ravel()
        vec_P0 = solve(np.eye(r**2) - Phi_kron, vec_Q)
        P_pred[0] = vec_P0.reshape(r, r)
        P_pred[0] = (P_pred[0] + P_pred[0].T) / 2
    except:
        P_pred[0] = np.eye(r) * 10
    
    # Forward pass
    for t in range(T):
        # Prediction error
        v = X[t] - Lambda @ F_pred[t]
        F_var = Lambda @ P_pred[t] @ Lambda.T + Sigma_e
        F_var = (F_var + F_var.T) / 2
        
        # Kalman gain
        try:
            K = P_pred[t] @ Lambda.T @ solve(F_var, np.eye(N))
        except:
            K = np.zeros((r, N))
        
        # Update
        F_filt[t] = F_pred[t] + K @ v
        P_filt[t] = P_pred[t] - K @ Lambda @ P_pred[t]
        P_filt[t] = (P_filt[t] + P_filt[t].T) / 2
        
        # Predict next
        if t < T - 1:
            F_pred[t+1] = Phi @ F_filt[t]
            P_pred[t+1] = Phi @ P_filt[t] @ Phi.T + Sigma_eta
            P_pred[t+1] = (P_pred[t+1] + P_pred[t+1].T) / 2
    
    # Step 2: Backward simulation
    F_draw = np.zeros((T, r))
    
    # Sample at T
    try:
        L = cholesky(P_filt[T-1], lower=True)
        F_draw[T-1] = F_filt[T-1] + L @ np.random.randn(r)
    except:
        F_draw[T-1] = F_filt[T-1]
    
    # Backward recursion
    for t in range(T-2, -1, -1):
        try:
            # Smoother gain
            J_t = P_filt[t] @ Phi.T @ solve(P_pred[t+1], np.eye(r))
            
            # Conditional mean and variance
            F_cond_mean = F_filt[t] + J_t @ (F_draw[t+1] - F_pred[t+1])
            P_cond = P_filt[t] - J_t @ P_pred[t+1] @ J_t.T
            P_cond = (P_cond + P_cond.T) / 2
            
            # Sample
            L = cholesky(P_cond, lower=True)
            F_draw[t] = F_cond_mean + L @ np.random.randn(r)
        except:
            F_draw[t] = F_filt[t]
    
    return F_draw


print("Simulation smoother implemented!")

### Conditional 2: Loadings Given Factors

In [ ]:
def draw_loadings(X, F, Sigma_e, priors):
    """
    Draw Lambda | X, F, Sigma_e from conditional normal.
    
    For each variable i:
        X_i = lambda_i' F + e_i
    
    This is Bayesian linear regression.
    
    Parameters
    ----------
    X : ndarray (T, N)
        Observations
    F : ndarray (T, r)
        Factors
    Sigma_e : ndarray (N, N)
        Measurement error covariance (diagonal)
    priors : BayesianPriors
        Prior specifications
        
    Returns
    -------
    Lambda : ndarray (N, r)
        Sampled loadings
    """
    T, N = X.shape
    r = F.shape[1]
    
    Lambda = np.zeros((N, r))
    
    # Precompute F'F
    FF = F.T @ F
    
    # Prior precision
    Sigma_0_inv = solve(priors.Sigma_lambda, np.eye(r))
    
    for i in range(N):
        # Likelihood precision: (1/sigma_e_i^2) * F'F
        sigma_e_i = Sigma_e[i, i]
        
        # Posterior precision = prior precision + likelihood precision
        Sigma_i_inv = Sigma_0_inv + FF / sigma_e_i
        Sigma_i = solve(Sigma_i_inv, np.eye(r))
        
        # Posterior mean
        mu_i = Sigma_i @ (Sigma_0_inv @ priors.mu_lambda + (F.T @ X[:, i]) / sigma_e_i)
        
        # Draw from posterior
        try:
            Lambda[i, :] = multivariate_normal.rvs(mean=mu_i, cov=Sigma_i)
        except:
            Lambda[i, :] = mu_i  # Fallback to mean
    
    return Lambda


print("Loading sampler implemented!")

### Conditional 3: Dynamics Given Factors

In [ ]:
def draw_dynamics(F, Sigma_eta, priors):
    """
    Draw Phi | F, Sigma_eta from conditional normal.
    
    For each factor i:
        F_it = phi_i' F_{t-1} + eta_it
    
    This is also Bayesian linear regression.
    
    Parameters
    ----------
    F : ndarray (T, r)
        Factors
    Sigma_eta : ndarray (r, r)
        Factor innovation covariance
    priors : BayesianPriors
        Prior specifications
        
    Returns
    -------
    Phi : ndarray (r, r)
        Sampled dynamics matrix
    """
    T, r = F.shape
    
    # Construct regression
    Y = F[1:, :]  # T-1 x r
    X_lag = F[:-1, :]  # T-1 x r
    
    Phi = np.zeros((r, r))
    
    # Precompute X'X
    XX = X_lag.T @ X_lag
    
    # Prior precision
    Sigma_phi_inv = solve(priors.Sigma_phi, np.eye(r))
    
    for i in range(r):
        # Likelihood precision
        sigma_eta_i = Sigma_eta[i, i]
        
        # Posterior precision
        Sigma_i_inv = Sigma_phi_inv + XX / sigma_eta_i
        Sigma_i = solve(Sigma_i_inv, np.eye(r))
        
        # Posterior mean
        mu_i = Sigma_i @ (Sigma_phi_inv @ priors.mu_phi + (X_lag.T @ Y[:, i]) / sigma_eta_i)
        
        # Draw
        try:
            Phi[i, :] = multivariate_normal.rvs(mean=mu_i, cov=Sigma_i)
        except:
            Phi[i, :] = mu_i
    
    return Phi


print("Dynamics sampler implemented!")

### Conditional 4: Measurement Variances

In [ ]:
def draw_sigma_e(X, F, Lambda, priors):
    """
    Draw Sigma_e | X, F, Lambda from inverse-gamma (diagonal).
    
    For each variable i:
        sigma_e_i^2 ~ IG(a_post, b_post)
    
    Parameters
    ----------
    X : ndarray (T, N)
        Observations
    F : ndarray (T, r)
        Factors
    Lambda : ndarray (N, r)
        Loadings
    priors : BayesianPriors
        Prior specifications
        
    Returns
    -------
    Sigma_e : ndarray (N, N)
        Sampled measurement error covariance (diagonal)
    """
    T, N = X.shape
    sigma_e_vec = np.zeros(N)
    
    for i in range(N):
        # Residuals
        resid = X[:, i] - F @ Lambda[i, :]
        SS = np.sum(resid**2)
        
        # Posterior parameters
        a_post = priors.a_e + T / 2
        b_post = priors.b_e + SS / 2
        
        # Draw from IG(a, b)
        sigma_e_vec[i] = invgamma.rvs(a=a_post, scale=b_post)
    
    return np.diag(sigma_e_vec)


print("Variance sampler implemented!")

## 4. Complete Gibbs Sampler

Now we combine all conditionals into the full MCMC algorithm.

In [ ]:
def gibbs_sampler_dfm(X, n_factors, n_iter=5000, burn_in=2000, thin=5,
                      informative_prior=False, verbose=True):
    """
    Gibbs sampler for Bayesian DFM.
    
    Parameters
    ----------
    X : ndarray (T, N)
        Observed data (demeaned)
    n_factors : int
        Number of factors
    n_iter : int
        Total MCMC iterations
    burn_in : int
        Burn-in period to discard
    thin : int
        Keep every thin-th sample
    informative_prior : bool
        Use informative vs diffuse priors
    verbose : bool
        Print progress
        
    Returns
    -------
    samples : dict
        Posterior samples
    """
    T, N = X.shape
    r = n_factors
    
    # Initialize priors
    priors = BayesianPriors(r, N, informative=informative_prior)
    
    if verbose:
        print("Initializing from PCA...")
    
    # Initialize parameters from PCA
    from sklearn.decomposition import PCA
    pca = PCA(n_components=r)
    F = pca.fit_transform(X)
    Lambda = pca.components_.T
    
    # Initialize dynamics (simple AR)
    Phi = np.eye(r) * 0.5
    
    # Initialize variances
    resid = X - F @ Lambda.T
    sigma_e_vec = np.var(resid, axis=0)
    Sigma_e = np.diag(sigma_e_vec)
    Sigma_eta = np.eye(r)  # Fixed for identification
    
    # Storage
    n_saved = (n_iter - burn_in) // thin
    samples = {
        'Lambda': np.zeros((n_saved, N, r)),
        'Phi': np.zeros((n_saved, r, r)),
        'sigma_e': np.zeros((n_saved, N)),
        'F': np.zeros((n_saved, T, r))
    }
    
    save_idx = 0
    
    if verbose:
        print(f"\nRunning Gibbs sampler for {n_iter} iterations...")
        print(f"Burn-in: {burn_in}, Thinning: {thin}")
        print(f"Will save {n_saved} samples\n")
    
    # MCMC loop
    for iteration in range(n_iter):
        # 1. Draw factors
        F = simulation_smoother_simple(X, Lambda, Phi, Sigma_e, Sigma_eta)
        
        # 2. Draw loadings
        Lambda = draw_loadings(X, F, Sigma_e, priors)
        
        # 3. Draw dynamics
        Phi = draw_dynamics(F, Sigma_eta, priors)
        
        # 4. Draw measurement variances
        Sigma_e = draw_sigma_e(X, F, Lambda, priors)
        
        # 5. Sigma_eta is fixed at I for identification
        
        # Save samples after burn-in
        if iteration >= burn_in and (iteration - burn_in) % thin == 0:
            samples['Lambda'][save_idx] = Lambda
            samples['Phi'][save_idx] = Phi
            samples['sigma_e'][save_idx] = np.diag(Sigma_e)
            samples['F'][save_idx] = F
            save_idx += 1
        
        if verbose and iteration % 500 == 0:
            print(f"Iteration {iteration}/{n_iter}")
    
    if verbose:
        print(f"\nGibbs sampler complete!")
        print(f"Saved {save_idx} posterior samples")
    
    return samples, priors


print("Complete Gibbs sampler ready!")

## 5. Application: Simulated Data

Let's test the Bayesian estimator on simulated data.

In [ ]:
# Purpose: Generate DFM data with known parameters

def simulate_dfm(T=150, N=20, r=2, seed=42):
    """Simulate DFM."""
    np.random.seed(seed)
    
    # True parameters
    Lambda_true = np.random.randn(N, r) * 0.8
    Phi_true = np.array([[0.7, 0.1], [0.1, 0.6]])
    sigma_e_true = np.ones(N) * 0.5
    
    # Simulate factors
    F = np.zeros((T, r))
    for t in range(1, T):
        F[t] = Phi_true @ F[t-1] + np.random.randn(r) * 0.5
    
    # Simulate observations
    X = F @ Lambda_true.T + np.random.randn(T, N) * sigma_e_true
    X = X - X.mean(axis=0)
    
    return X, F, Lambda_true, Phi_true, sigma_e_true


# Generate data
print("Generating simulated data...\n")
X, F_true, Lambda_true, Phi_true, sigma_e_true = simulate_dfm(T=150, N=20, r=2)

print(f"Data dimensions: T={X.shape[0]}, N={X.shape[1]}")
print(f"\nTrue Phi:")
print(Phi_true)
print(f"\nMean true sigma_e: {sigma_e_true.mean():.3f}")

### Run Gibbs Sampler

In [ ]:
# Purpose: Fit Bayesian DFM via MCMC

samples, priors = gibbs_sampler_dfm(
    X,
    n_factors=2,
    n_iter=3000,
    burn_in=1000,
    thin=2,
    informative_prior=False,
    verbose=True
)

## 6. Posterior Analysis

Now we analyze the posterior samples.

In [ ]:
# Purpose: Compute posterior summaries

# Posterior means
Phi_post_mean = samples['Phi'].mean(axis=0)
Lambda_post_mean = samples['Lambda'].mean(axis=0)
sigma_e_post_mean = samples['sigma_e'].mean(axis=0)

# Posterior standard deviations
Phi_post_std = samples['Phi'].std(axis=0)

# 95% credible intervals
Phi_lower = np.percentile(samples['Phi'], 2.5, axis=0)
Phi_upper = np.percentile(samples['Phi'], 97.5, axis=0)

print("="*60)
print("POSTERIOR SUMMARIES")
print("="*60)

print("\nPosterior mean of Phi:")
print(Phi_post_mean)

print("\nTrue Phi:")
print(Phi_true)

print("\nPosterior standard deviation of Phi:")
print(Phi_post_std)

print("\n95% Credible Intervals for Phi:")
for i in range(2):
    for j in range(2):
        print(f"  Phi[{i},{j}]: [{Phi_lower[i,j]:.3f}, {Phi_upper[i,j]:.3f}]  (true: {Phi_true[i,j]:.3f})")

print(f"\nMean posterior sigma_e: {sigma_e_post_mean.mean():.3f}")
print(f"True mean sigma_e: {sigma_e_true.mean():.3f}")

print("="*60)

### Trace Plots: Assess Convergence

In [ ]:
# Purpose: Visualize MCMC traces to assess mixing

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Phi[0,0]
ax = axes[0, 0]
ax.plot(samples['Phi'][:, 0, 0], linewidth=0.5, alpha=0.7)
ax.axhline(Phi_true[0, 0], color='red', linestyle='--', linewidth=2, label='True value')
ax.axhline(Phi_post_mean[0, 0], color='green', linestyle=':', linewidth=2, label='Posterior mean')
ax.set_title('Trace Plot: Phi[0,0]', fontsize=12, fontweight='bold')
ax.set_xlabel('Iteration')
ax.set_ylabel('Value')
ax.legend()
ax.grid(True, alpha=0.3)

# Phi[1,1]
ax = axes[0, 1]
ax.plot(samples['Phi'][:, 1, 1], linewidth=0.5, alpha=0.7, color='orange')
ax.axhline(Phi_true[1, 1], color='red', linestyle='--', linewidth=2)
ax.axhline(Phi_post_mean[1, 1], color='green', linestyle=':', linewidth=2)
ax.set_title('Trace Plot: Phi[1,1]', fontsize=12, fontweight='bold')
ax.set_xlabel('Iteration')
ax.set_ylabel('Value')
ax.grid(True, alpha=0.3)

# sigma_e[0]
ax = axes[1, 0]
ax.plot(samples['sigma_e'][:, 0], linewidth=0.5, alpha=0.7, color='green')
ax.axhline(sigma_e_true[0], color='red', linestyle='--', linewidth=2)
ax.set_title('Trace Plot: sigma_e[0]', fontsize=12, fontweight='bold')
ax.set_xlabel('Iteration')
ax.set_ylabel('Value')
ax.grid(True, alpha=0.3)

# Lambda[0,0]
ax = axes[1, 1]
ax.plot(samples['Lambda'][:, 0, 0], linewidth=0.5, alpha=0.7, color='purple')
ax.axhline(Lambda_true[0, 0], color='red', linestyle='--', linewidth=2)
ax.set_title('Trace Plot: Lambda[0,0]', fontsize=12, fontweight='bold')
ax.set_xlabel('Iteration')
ax.set_ylabel('Value')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nGood mixing: Trace should look like 'white noise' around the mean.")
print("No trends or patterns indicate good convergence.")

### Posterior Distributions

In [ ]:
# Purpose: Visualize posterior distributions

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Phi[0,0] histogram
ax = axes[0, 0]
ax.hist(samples['Phi'][:, 0, 0], bins=30, density=True, alpha=0.7, edgecolor='black')
ax.axvline(Phi_true[0, 0], color='red', linestyle='--', linewidth=2.5, label='True value')
ax.axvline(Phi_post_mean[0, 0], color='green', linestyle=':', linewidth=2.5, label='Posterior mean')
ax.set_title('Posterior: Phi[0,0]', fontsize=12, fontweight='bold')
ax.set_xlabel('Value')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Phi[1,1] histogram
ax = axes[0, 1]
ax.hist(samples['Phi'][:, 1, 1], bins=30, density=True, alpha=0.7, color='orange', edgecolor='black')
ax.axvline(Phi_true[1, 1], color='red', linestyle='--', linewidth=2.5)
ax.axvline(Phi_post_mean[1, 1], color='green', linestyle=':', linewidth=2.5)
ax.set_title('Posterior: Phi[1,1]', fontsize=12, fontweight='bold')
ax.set_xlabel('Value')
ax.set_ylabel('Density')
ax.grid(True, alpha=0.3, axis='y')

# sigma_e[0] histogram
ax = axes[1, 0]
ax.hist(samples['sigma_e'][:, 0], bins=30, density=True, alpha=0.7, color='green', edgecolor='black')
ax.axvline(sigma_e_true[0], color='red', linestyle='--', linewidth=2.5)
ax.set_title('Posterior: sigma_e[0]', fontsize=12, fontweight='bold')
ax.set_xlabel('Value')
ax.set_ylabel('Density')
ax.grid(True, alpha=0.3, axis='y')

# Credible interval coverage
ax = axes[1, 1]
param_names = ['Phi[0,0]', 'Phi[0,1]', 'Phi[1,0]', 'Phi[1,1]']
coverage = []
for i in range(2):
    for j in range(2):
        lower = Phi_lower[i, j]
        upper = Phi_upper[i, j]
        true_val = Phi_true[i, j]
        covered = 1 if lower <= true_val <= upper else 0
        coverage.append(covered)

colors = ['green' if c else 'red' for c in coverage]
ax.bar(param_names, coverage, color=colors, alpha=0.7)
ax.set_title('95% CI Coverage', fontsize=12, fontweight='bold')
ax.set_ylabel('Covered (1) or Not (0)')
ax.set_ylim([0, 1.2])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nCI coverage: {sum(coverage)}/4 parameters covered by 95% credible intervals")

## Summary

### Key Takeaways

1. **Bayesian estimation provides full uncertainty quantification:** We get distributions, not just point estimates

2. **Priors regularize estimates:** Informative priors prevent overfitting and incorporate domain knowledge

3. **Gibbs sampler enables inference:** By sampling from conditional distributions, we obtain joint posterior samples

4. **Simulation smoother draws latent factors:** This is the E-step analog in Bayesian inference

5. **Credible intervals have natural interpretation:** "95% probability parameter is in this interval" (unlike frequentist CIs)

6. **MCMC diagnostics are essential:** Trace plots, convergence tests ensure valid inference

7. **Computational cost is higher than MLE:** But we gain:
   - Natural uncertainty quantification
   - Regularization through priors
   - Handling of complex constraints
   - Posterior predictive distributions

### Bayesian vs Frequentist

| Aspect | Frequentist (MLE/EM) | Bayesian (MCMC) |
|--------|----------------------|----------------|
| Output | Point estimates | Full distributions |
| Uncertainty | Asymptotic std errors | Posterior variance |
| Prior info | Not incorporated | Natural via priors |
| Regularization | Ad-hoc penalties | Automatic via priors |
| Speed | Faster | Slower |
| Interpretation | "True parameter" | "Belief given data" |

### When to Use Bayesian

- **Need uncertainty quantification:** Policy decisions, risk assessment
- **Have prior information:** Economic theory, previous studies
- **Small samples:** Priors prevent overfitting
- **Complex constraints:** Sign restrictions, ordering
- **Forecasting:** Posterior predictive distribution

### What's Next

Module 5 introduces **mixed-frequency data**, where variables are observed at different frequencies (e.g., monthly and quarterly). This is crucial for real-time nowcasting and policy analysis.

### Additional Resources

- Carter & Kohn (1994): "On Gibbs Sampling for State Space Models"
- Kim & Nelson (1999): *State-Space Models with Regime Switching*
- Module guide: `guides/03_bayesian_dfm.md`

---

**Completion:** Module 4 complete! You now master three estimation approaches: PCA, EM, and Bayesian MCMC.